In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.gold
COMMENT 'Capa Gold procesados'

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_tipo_predio")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_tipo_predio (
  tipo_predio_id INT,
  tipo_predio_descripcion STRING
)
USING DELTA
""")

In [0]:
import pandas as pd

df = spark.table("workspace.silver.predios_registro").toPandas()

In [0]:
df.columns

In [0]:
from pyspark.sql import functions as F

dim = df[[
    "tipo_predio_descripcion"
]].dropna().drop_duplicates().reset_index(drop=True)

dim["tipo_predio_id"] = dim.index + 1
dim = dim.sort_values("tipo_predio_id").reset_index(drop=True)
dim = dim[[
    "tipo_predio_id",
    "tipo_predio_descripcion"
]]

df_spark = spark.createDataFrame(dim)

df_spark = df_spark.withColumn(
    "tipo_predio_id",
    F.col("tipo_predio_id").cast("int")
)

In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_tipo_predio")